# 26 — Job Recommender (v1)

**Purpose**  
This notebook builds the first functional job recommender for Chapter 4 and validates that it can run end-to-end using the canonical Chapter 4 context payload. It serves two roles:  
1) a **development workspace** to iterate on thresholds, bucketing, and ranking logic, and  
2) a **verification harness** that confirms salary-prediction features are aligned and attach cleanly to recommended jobs.

---

## What the notebook produces

### Primary outputs
- **Candidate job tables** filtered by suitability and split into accessibility buckets:
  - `best_now` shortlist (top *N*)
  - `stretch` shortlist (top *N*)
- **Per-job salary predictions** (`pred_sal`) generated using:
  - the persisted Chapter 1 Salary Response Model, and
  - the Chapter 4 salary feature matrix constructed from candidate job codes + broadcasted user skill PCs
- **Bucket-level salary diagnostics**:
  - mean expected salary (`sal_mean`) vs mean predicted salary (`pred_sal`)
  - deltas within and across buckets (e.g., “salary jump” from best-now → stretch)

### Secondary outputs (console diagnostics)
- Counts after each filter stage (suitability filtering, bucket sizes)
- Warnings when constraints produce too few jobs in a bucket
- Smoke checks confirming `pred_sal` has no missing values and aligns to candidate rows

---

## Workflow (step-by-step)

### 1) Load Chapter 4 context (single entrypoint)
The notebook calls `load_ch4_context()` to obtain a canonical payload that includes:
- `candidates_df` with suitability + competitiveness
- the loaded salary model (`salary_model`)
- the candidate-level salary design matrix (`user_salary_model_features`)

This prevents recomputing upstream artefacts and ensures all downstream steps operate on aligned, stable inputs.

### 2) Predict user-implied salary for every candidate job
Using the salary model and the feature matrix, the notebook generates:
- `pred_sal` for every row in `candidates_df`

A minimal alignment check confirms:
- number of predictions equals number of candidate rows
- no missing predictions are introduced

### 3) Retrieve eligible jobs using suitability thresholding
The notebook applies a base suitability threshold (`S_min_base`).  
If fewer than `N_target` jobs remain, it falls back to a lower threshold (`S_min_floor`).  
If the candidate set is still too small, the notebook raises an explicit failure to avoid silently recommending low-signal results.

### 4) Split into accessibility buckets using competitiveness
Eligible jobs are bucketed into:
- `best_now` if `competitiveness_index <= C_max`
- `stretch` otherwise

The notebook reports bucket sizes and emits warnings if bucket counts fall below minimums (guardrails against misleading recommendation output).

### 5) Rerank within the eligible pool
A combined score is computed:

`score = suitability - alpha * competitiveness_index`

Jobs are sorted deterministically using:
- `score` (primary)
- `suitability` (secondary)
- `competitiveness_index` (tertiary)
- `job_id` (tie-breaker)

Then the notebook prints the top-*N* jobs per bucket with a compact set of columns for inspection.

### 6) Compare expected vs predicted salary in the shortlists
For each bucket, the notebook computes:
- mean `sal_mean` (expected market salary in the dataset context)
- mean `pred_sal` (user-implied salary from skills + job context)
- delta = expected − predicted

It also computes the change in this delta between buckets (a coarse “stretch premium” diagnostic).

---

## How this notebook maps to `src/`
This notebook is the prototype for `features/job_recommender.py`.  
After the notebook stabilises, the final logic is migrated to `src/` with:
- the same thresholds and contract,
- a structured return payload (tables + summaries),
- and minimal print statements (prints only in notebook, not required in production).

---

## Scope notes (v1)
- This notebook produces job shortlists and salary diagnostics only.  
- Explanation narratives (why job / why skill), upskilling recommendations, and what-if simulation are intentionally deferred to later Chapter 4 modules.


## Libraries

In [14]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path
import joblib

## Paths

In [15]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [16]:
from src.job_intel.features.artefacts_ch4 import load_ch4_context

## Candidate set

In [17]:
skill_text= "python, sql, bayesian, communication, problem-solving, research, publication, causal inference, statistical modelling," \
"r, ecology, visualisaion, ggpplot, seaborn, numpy, pandas, git, github, microsoft office, phd, neural networks, excell, teamwork, team member, cloud, aws" \
"pca, recommender systems, shiny app, shiny, technical writting, scientific research"
current_state= ("ALL")
job_title_family = "data_scientist"
job_title_rich= None
target_sectors = None
salary_target = 200000
explain_skills = True

In [18]:
candidate = load_ch4_context(skill_text=skill_text,
                             current_state=current_state,
                             job_title_family=job_title_family,
                             job_title_rich=job_title_rich,
                             target_sectors=target_sectors,
                             salary_target=salary_target,
                             explain_skills=explain_skills)

can_df = candidate['candidates_df']
sal = candidate['salary_model']
feat = candidate['user_salary_model_features']

y_hat = sal.predict(feat)
can_df['pred_sal'] = y_hat

In [32]:
candidate['profile']['derived']['skill_explanations']

{'core_programming__basic': ['python', 'r'],
 'core_programming__intermediate': [],
 'core_programming__advanced': [],
 'data_engineering_pipelines__basic': [],
 'data_engineering_pipelines__intermediate': [],
 'data_engineering_pipelines__advanced': [],
 'ml_ai__basic': ['statistical modelling'],
 'ml_ai__intermediate': ['recommender', 'recommender systems', 'bayesian'],
 'ml_ai__advanced': ['neural networks'],
 'analytics_stats__basic': ['pandas', 'numpy'],
 'analytics_stats__intermediate': [],
 'analytics_stats__advanced': ['causal inference'],
 'bi_viz__basic': ['seaborn'],
 'bi_viz__intermediate': ['shiny'],
 'bi_viz__advanced': [],
 'cloud__basic': ['cloud'],
 'cloud__intermediate': [],
 'cloud__advanced': [],
 'db_storage__basic': ['sql'],
 'db_storage__intermediate': [],
 'db_storage__advanced': [],
 'productivity_workflow__basic': [],
 'productivity_workflow__intermediate': ['git', 'github'],
 'productivity_workflow__advanced': [],
 'soft_skills__core': ['communication', 'team

### Preddict salary

In [19]:
y_hat = sal.predict(feat)
can_df['pred_sal'] = y_hat

In [20]:
# Checks

print(f'Acceptance shape: {len(y_hat) == len(can_df)}')
print(f'Acceptance alignment: {can_df["pred_sal"].isna().sum() == 0}')


Acceptance shape: True
Acceptance alignment: True


### Apply suitabillity threshold

In [21]:
S_min_base = 0.70
S_min_floor = 0.60
N_target = 50

try_base = can_df[can_df["suitability"] >= S_min_base]
try_floor = can_df[can_df["suitability"] >= S_min_floor]

print(f"Applying the suitability index threshold {S_min_base}...")

if len(try_base) < N_target:
    print(
        f"Total number of jobs returned is too low (<{N_target}); "
        f"trying a lower suitability threshold to {S_min_floor}"
    )

    if len(try_floor) < N_target:
        raise ValueError(
            f"Total number of jobs returned is still too low (<{N_target}) after lowering "
            f"the suitability threshold to {S_min_floor}. Reduce your constraints or switch to upskilling."
        )

    candidate_jobs = try_floor.copy()
    print(f"* Returning jobs after applying {S_min_floor} threshold.")
    print(f"* Suitable jobs identified = {len(candidate_jobs)}.")
    print(f"* {len(can_df) - len(candidate_jobs)} filtered out due to low suitability.")
else:
    candidate_jobs = try_base.copy()
    print(f"* Returning jobs after applying {S_min_base} threshold.")
    print(f"* Suitable jobs identified = {len(candidate_jobs)}.")
    print(f"* {len(can_df) - len(candidate_jobs)} filtered out due to low suitability.")


Applying the suitability index threshold 0.7...
* Returning jobs after applying 0.7 threshold.
* Suitable jobs identified = 287.
* 1015 filtered out due to low suitability.


### Apply competitiveness buckets

In [22]:
C_max = 0.50
min_bucket_size_bestnow = 10
min_bucket_size_stretch = 5

print('Applying competitiveness filter to separate remaining jobs into 2 buckets: "best-now" and "stretch".')

candidate_jobs["competitiveness_bucket"] = np.where(
    candidate_jobs["competitiveness_index"] <= C_max, "best_now", "stretch"
)

vc = candidate_jobs["competitiveness_bucket"].value_counts()
best_now = vc.get("best_now", 0)
stretch = vc.get("stretch", 0)

print(f'Number of "Best-now" jobs = {best_now}')
print(f'Number of "Stretch" jobs = {stretch}')

if best_now <= min_bucket_size_bestnow:
    print('WARNING: Low number of "best-now" options. Consider relaxing constraints or using upskilling.')

if stretch <= min_bucket_size_stretch:
    print('WARNING: Low number of "stretch" options. Consider relaxing constraints or using upskilling.')


Applying competitiveness filter to separate remaining jobs into 2 buckets: "best-now" and "stretch".
Number of "Best-now" jobs = 274
Number of "Stretch" jobs = 13


### Rank jobs

In [23]:
# score = suitability - alpha * competitiveness_index
alpha = 0.5
top_n_best = 10
top_n_stretch = 5

candidate_jobs['score'] = candidate_jobs['suitability'] - (alpha * candidate_jobs['competitiveness_index'])

candidate_jobs = candidate_jobs.sort_values(by=['score', 'suitability', 'competitiveness_index', 'job_id'], ascending=[False, False, True, True])

top_best_now = candidate_jobs[candidate_jobs['competitiveness_bucket'] == 'best_now'].set_index("job_id")[[
                                                                                                            'Size',
                                                                                                            'Sector',
                                                                                                            'Industry',
                                                                                                            'state',
                                                                                                            'title_rich',
                                                                                                            'sal_mean',
                                                                                                            'pred_sal',
                                                                                                            'suitability',
                                                                                                            'competitiveness_index',
                                                                                                            'score']]
print(f'Top {top_n_best} "best-now" jobs:\n')
print(top_best_now.head(top_n_best))

top_stretch = candidate_jobs[candidate_jobs['competitiveness_bucket'] == 'stretch'].set_index("job_id")[[
                                                                                                        'Size',
                                                                                                        'Sector',
                                                                                                        'Industry',
                                                                                                        'state',
                                                                                                        'title_rich',
                                                                                                        'sal_mean',
                                                                                                        'pred_sal',
                                                                                                        'suitability',
                                                                                                        'competitiveness_index',
                                                                                                        'score']]
print(f'Top {top_n_stretch} "stretch" jobs:\n')
print(top_stretch.head(top_n_stretch))

top_best_now["sal_mean"] = pd.to_numeric(top_best_now["sal_mean"], errors="coerce").head(top_n_best)
top_best_now["pred_sal"] = pd.to_numeric(top_best_now["pred_sal"], errors="coerce").head(top_n_best)
top_stretch["sal_mean"] = pd.to_numeric(top_stretch["sal_mean"], errors="coerce").head(top_n_stretch)
top_stretch["pred_sal"] = pd.to_numeric(top_stretch["pred_sal"], errors="coerce").head(top_n_stretch)


sal_mean_expected_best = top_best_now['sal_mean'].mean()
sal_mean_pred_best = top_best_now['pred_sal'].mean()
delta_best = sal_mean_expected_best - sal_mean_pred_best
sal_mean_expected_strecth = top_stretch['sal_mean'].mean()
sal_mean_pred_strecth = top_stretch['pred_sal'].mean()
delta_stretch = sal_mean_expected_strecth - sal_mean_pred_strecth

mean_sal_jump = delta_stretch - delta_best

print('Best-now salary comparison:')
print('---------------------------')
print(f'Expected mean salary: {sal_mean_expected_best:.2f}')
print(f'Predicted mean salary (based on your skills): {sal_mean_pred_best:.2f}')
print(f'Delta (expected - predicted): {delta_best:.2f}')

print('\n')

print('Stretch salary comparison:')
print('---------------------------')
print(f'Expected mean salary: {sal_mean_expected_strecth:.2f}')
print(f'Predicted mean salary (based on your skills): {sal_mean_pred_strecth:.2f}')
print(f'Delta (expected - predicted): {delta_stretch:.2f}')

print('\n')

print(f'Mean salary jump from "Best-now" to "Stretch" jobs: {(mean_sal_jump):.2f}')

Top 10 "best-now" jobs:

                          Size                      Sector  \
job_id                                                       
821     1001 to 5000 employees      Information Technology   
289          1 to 50 employees      Information Technology   
112     1001 to 5000 employees           Business Services   
797           10000+ employees                   Insurance   
744     1001 to 5000 employees      Information Technology   
895                    Unknown                     Unknown   
1967     501 to 1000 employees           Business Services   
267          1 to 50 employees                     Unknown   
3387     501 to 1000 employees                   Education   
4994    1001 to 5000 employees  Transportation & Logistics   

                               Industry state                   title_rich  \
job_id                                                                       
821                         IT Services    IL  general_data_data_scientist

### Package results

In [24]:
params = {  'S_min_base' : 0.70,
            'S_min_floor' : 0.60,
            'N_target' : 50,
            'C_max' : 0.50,
            'min_bucket_size_bestnow' : 10,
            'min_bucket_size_stretch' : 5,
            'alpha' : 0.5,
            'top_n_best' : 10,
            'top_n_stretch' : 5}

counts = {'best_now':best_now,
          'stretch':stretch,
          'candidate_jobs':len(candidate_jobs),
          }

tables = {'candidate_jobs':candidate_jobs,
          'top_best_now':top_best_now,
          'top_stretch':top_stretch    
}

salary = {'sal_mean_expected_best':sal_mean_expected_best,
          'sal_mean_pred_best':sal_mean_pred_best,
          'delta_best':delta_best,
          'sal_mean_expected_strecth':sal_mean_expected_strecth,
          'sal_mean_pred_strecth':sal_mean_pred_strecth,
          'delta_stretch':delta_stretch,
          'mean_sal_jump':mean_sal_jump
}

result = {'params': params,
          'counts': counts,
          'tables':tables,
          'salary_summary': salary}

## Test src

In [25]:
from src.job_intel.features.job_recommender import job_recommender
result = job_recommender(skill_text=skill_text,
                             current_state=current_state,
                             job_title_family=job_title_family,
                             job_title_rich=job_title_rich,
                             target_sectors=target_sectors,
                             salary_target=salary_target,
                             explain_skills=explain_skills,
                             verbose=True)

Predicting salary based on input skills...
Acceptance shape test passed: True
Acceptance alignment test passed: True
Applying the suitability index threshold 0.7...
* Returning jobs after applying s_min=0.7.
* Suitable jobs identified = 287.
* 1015 filtered out due to low suitability.
Applying competitiveness filter into 2 buckets: "best-now" and "stretch".
Number of "Best-now" jobs = 274
Number of "Stretch" jobs = 13
Computing ranking score based on suitability and competitiveness.
Top 10 "best-now" jobs:

                          Size                      Sector  \
job_id                                                       
821     1001 to 5000 employees      Information Technology   
289          1 to 50 employees      Information Technology   
112     1001 to 5000 employees           Business Services   
797           10000+ employees                   Insurance   
744     1001 to 5000 employees      Information Technology   
895                    Unknown                     Un

In [26]:
(candidate_jobs[['competitiveness_bucket','sal_mean', 'pred_sal']] == result['tables']['candidate_jobs'][['competitiveness_bucket','sal_mean', 'pred_sal']]).mean()

competitiveness_bucket    1.0
sal_mean                  1.0
pred_sal                  1.0
dtype: float64

# === End of Notebook ===